# Homework 1 — Scientific Paper Processing with spaCy

Today's first experiment: extract text from PDF pages while preserving page boundaries.

We will inspect the output on the first, middle, and last pages, then record errors in `notes/experiment_log.md`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from scipaper.extraction import extract_pdf, document_statistics, save_extracted_text

## PDF paths

The course PDFs are expected in the repository-local `data/papers/` directory.


In [ ]:
PAPER_DIR = PROJECT_ROOT / 'data' / 'papers'
PDF_FILES = [
    PAPER_DIR / 'AttentionYouNeed.pdf',
    PAPER_DIR / '1406.1078v3.pdf',
    PAPER_DIR / '1409.0473v7.pdf',
    PAPER_DIR / '1412.3555v1.pdf',
    PAPER_DIR / '1601.06733v7.pdf',
    PAPER_DIR / '1607.06450v1.pdf',
    PAPER_DIR / '1610.02357v3.pdf',
    PAPER_DIR / '1703.03906v2.pdf',
    PAPER_DIR / 'N16-1024.pdf',
]
SORT_BLOCKS = True  # Initial run; compare with False on the same pages.
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'extracted_text'

missing = [str(path) for path in PDF_FILES if not path.is_file()]
if missing:
    print('PDF paths not found:', missing)
else:
    print(f'{len(PDF_FILES)} PDFs ready from {PAPER_DIR}')

In [ ]:
results = []
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    text_path = OUTPUT_DIR / f'{pdf_path.stem}.txt'
    save_extracted_text(pages, text_path)
    results.append({'paper': pdf_path.stem, **document_statistics(pages), 'text_file': str(text_path)})

results

In [ ]:
# Inspect short excerpts from the first, middle, and last pages.
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    selected = sorted({0, len(pages) // 2, len(pages) - 1})
    print(f'\n### {pdf_path.name}')
    for index in selected:
        print(f'\n--- page {pages[index]["page"]} ---\n{pages[index]["text"][:1200]}')

## Task 2 — Section and subsection detection

This experiment detects headings from PDF layout blocks. The first rule uses numbered or lettered section IDs together with font emphasis or a size increase. A second rule handles compact small-caps headings whose PDFs encode the text in a regular font. Common unnumbered labels such as Abstract, Acknowledgments, and References are also included. The detector records the one-based PDF page and does not contain paper-specific section lists.

The results are candidates for review, not guaranteed ground truth. We manually inspected representative pages from different layouts and recorded false positives, missed headings, and limitations in `notes/experiment_log.md`. The sample is too small to claim perfect recall or precision across every page.

In [ ]:
from scipaper.sections import detect_sections

section_results = {
    pdf_path.stem: detect_sections(pdf_path)
    for pdf_path in PDF_FILES
}

for paper_id, sections in section_results.items():
    print(f"{paper_id}: {len(sections)} heading candidates")
    for section in sections:
        section_id = section["section_id"] or "(unnumbered)"
        print(f"  page {section['page']:>2} | {section_id:<7} | {section['title']}")
    print()

### Manual review checkpoint

Useful pages to compare against the rendered PDFs include `1409.0473v7` page 2 (small-caps subsection 2.1), `1610.02357v3` page 2 (wrapped subsection 1.2), `1703.03906v2` page 3 (two-column subsections 3.2 and 3.3), and `N16-1024` page 2 (sections 2, 3, and 3.1). Review the visual source before treating any candidate as correct; the experiment log records what was checked and what remains uncertain.